# Using Tabular Foundation Model for Classification
### Model used: AutoGluon's **Mitra** tabular foundation model (https://huggingface.co/autogluon/mitra-classifier)
### Use case: Classifying flowers into their species based on their sepal and petal length and width.

### Step 0 — Install
Install AutoGluon with the Mitra extra (commented out; run once if not already installed).

In [ ]:
#!pip install autogluon.tabular[mitra]

### Step 1 — Imports
Bring in pandas, AutoGluon's `TabularDataset`/`TabularPredictor`, the train/test splitter, and the iris dataset loader.

In [ ]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

### Step 2 — Load the data
Fetch the iris dataset and build a DataFrame, adding the categorical `target` column (setosa, versicolor and virginica) we want to classify.

In [ ]:
iris_data = load_iris()
iris_df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
iris_df['target'] = iris_data.target  # setosa, versicolor and virginica

print(f"Dataset shape: {iris_df.shape}")
print(f"Classes: {iris_df['target'].nunique()} -> {sorted(iris_df['target'].unique())}")
iris_df.head()

### Step 3 — Split Datasets
Create an 80/20 train/test split. We `stratify` on the target so each class is proportionally represented in both sets.

In [ ]:
iris_train, iris_test = train_test_split(
    iris_df, test_size=0.2, random_state=42, stratify=iris_df['target']
)
iris_test = iris_test.drop("target", axis=1)
print(iris_train.shape, iris_test.shape)

### Step 4 — Save splits to CSV
Persist the train and test sets to disk for reuse / reproducibility.

In [ ]:
iris_train.to_csv('iris_train.csv', index=False)
iris_test.to_csv('iris_test.csv', index=False)

### Step 5 — Wrap in TabularDataset
Convert the DataFrames into AutoGluon `TabularDataset` objects (a thin pandas wrapper AutoGluon expects).

In [ ]:
iris_train_data = TabularDataset(iris_train)
iris_test_data = TabularDataset(iris_test)

### Step 6 — Create the predictor
Define a classification `TabularPredictor` for the `target` column and choose where the model is saved. AutoGluon infers the problem type (multiclass classification) automatically, but we set it explicitly for clarity.

In [ ]:
mitra_clf_predictor = TabularPredictor(
    label='target',
    path='./mitra_classifier_model',
    problem_type='multiclass'
)

### Step 7 — Train Mitra
Fit the Mitra foundation model (in-context, no fine-tuning) on the training rows. This step does not train from scratch — it prepares your labeled data as context for the pretrained model.

In [ ]:
mitra_clf_predictor.fit(
    iris_train_data,
    hyperparameters={
        'MITRA': {'fine_tune': False}
    },
)

### Step 8 — Predict
Generate class predictions on the test set. The result is a pandas Series aligned to the test index.

In [ ]:
predictions = mitra_clf_predictor.predict(iris_test_data)

### Step 9 — View predictions
Display the predicted classes, and the per-class probabilities from `predict_proba`.

In [ ]:
print("Predicted classes:")
print(predictions.head(10))

print("\nClass probabilities:")
probabilities = mitra_clf_predictor.predict_proba(iris_test_data)
probabilities.head()